# Argon Gas/Liquid Simulation
> NAME: Harrison B. Prosper<br>
> DATE: March 2026 

## Learning objective

To learn how to simulate a cloud of interacting neutral argon atoms confined to  a **spherical container**. 

## Tips

  * Use __esc r__ to disable a cell
  * Use __esc y__ to reactivate it
  * Use __esc m__ to go to markdown mode. **Markdown** is the typesetting language used in jupyter notebooks.
  * In a markdown cell, double tap the mouse or glide pad (on your laptop) to go to edit mode. 
  * Shift + return to execute a cell (including markdown cells).
  * If the equations don't typeset, try double tapping the cell again, and re-execute it.

In [2]:
import os, sys

# Array manipulation and linear algebra
import numpy as np

# 3D animation system
import vpython as vp

from comphyslab.graphics import Bag, Sim, Controls, \
Scene, Zoom, CoordinateSystem, I, J, K

from comphyslab.vectors import unit, magnitude

## Intersection of a line with a sphere

The equation of a straight line in 3D space is given by
\begin{align}
    \vec{r} & = \vec{r}_0 + t \hat{u}.\tag{1}
\end{align}

The equation of a sphere of radius $d$, centered at the origin, is simply
\begin{align}
    r = d, \tag{2}
\end{align}
where $r = ||\vec{r}||$, that is, the magnitude of $\vec{r}$.

![sphere/line intersection](sphere_line.png)

**Not graded**

Derive an expression for the two intersection points of a line with a sphere by solving Eqs.(1) and (2) *simultaneously*.

*Hint*: Take the dot product of Eq.(1) with itself, then use Eq.(2).

In [3]:
def reflect(v, n):
    '''
    v:    Velocity of particles, shape (N,3) with N >= 1
    n:    Unit vector defining orientation of plane, shape (3,)
    '''
    # vdotn must be of the same dimensions as v and n, that is, a 2D array.
    # We use broadcasting to achieve this.
    
    many  = len(v) > 1
    vdotn = (v*n).sum(axis=-1)[:, None] if many else  (v*n).sum(axis=-1) 

    u = v - 2 * vdotn * n  # (N,3)
    
    return u

In [4]:
def line_sphere_point(c, u, d):
    '''
    c:    Fixed point on line(s), shape (N,3)
    u:    Unit vector defining the direction of line(s), shape (N,3)
    d:    Radius of sphere centered at origin
    '''
    cc = (c*c).sum(axis=-1) # c dot c
    uc = (u*c).sum(axis=-1) # u dot c
    
    s  = uc**2 - cc + d**2
    
    # s >= 0 if the line crosses the sphere
    crosses = s >= 0
 
    # Two possible intersection points defined by the two
    # scalars t1 and t2. Protect against possible tiny negative 
    # values due to floating point rounding errors.
    q = np.sqrt(np.clip(s, 0, None))
    t1 =-uc + q
    t2 =-uc - q

    # Since we are within the sphere, we choose the greater of t1 and t2    
    t = np.where(t1 > t2, t1, t2)

    # vdotn must be of the same dimensions as v and n, that is, a 2D array.
    # We use broadcasting to achieve this.
    is2D = len(c) > 1
    b = c + t[:, None] * u if is2D else c + t * u
    return b

## Define initial state of system

### Argon atom properties

| **Quantity** | **Symbol** | **Value** |
|----------|----------|----------|
| Mass     | $m$       | $6.69\times 10^{-26}\,\text{kg}$ |
| Distance scale | $\sigma$    | $3.40\times 10^{-10}\,\text{m}$  |
| Characteristic time scale     | $t_c$       | $2.16\times 10^{-12}\,\text{s}$  |
| Characteristic speed    | $v_c$       | $157.38\, \text{m/s}$   |

### Conditions

  * Temperature: $T = 273\,\text{K}$ (0 Celcius).
  * density: $\rho = 50.0\text{ kg}/\text{m}^3$
  * Average speed of atoms: $2.6$ in units of $v_c$.
  * Container radius: $R = 13.414$ in units of $\sigma$.

# Exercise 5.1 [5 pt]

How many atoms of argon would need to be placed within the spherical container to get the desired density $\rho$ of $50 \text{ kg}/\text{m}^3$?

In [5]:
def define_initial_state():
    # --------------------------------------------
    # Create empty bag
    # --------------------------------------------
    bg = Bag()

    # --------------------------------------------
    # Simulation parameters
    # --------------------------------------------
    bg.dt = 5.0e-2              # Time increment (units of t0)
    
    # --------------------------------------------
    # Container parameters
    # --------------------------------------------
    bg.R = 13.414               # Radius of container (units of sigma)

    # --------------------------------------------
    # Parameters of atoms
    # --------------------------------------------
    bg.N = 5                    # Number of argon atoms
    bg.m = np.ones(bg.N)        # Mass of argon atom (in units of m)

    # Initial positions of atoms
    rhat = unit(np.random.uniform(-1, 1, 3*bg.N).reshape(bg.N, 3))
    bg.r = np.random.uniform(0.05, 0.95*bg.R) * rhat 
    bg.r0= np.zeros_like(bg.r)  # reserve buffer for use in update
    
    # Initial velocities of atoms
    vhat = unit(np.random.uniform(-1, 1, 3*bg.N).reshape(bg.N, 3))
    bg.v = 2.6 * vhat           # (units of vc)
    bg.v0= np.zeros_like(bg.v)  # reserve buffer for use in update

    # Remove center of mass motion
    if (len(bg.v.shape) > 1):
        if (bg.v.shape[0] > 1): 
            bg.v -= bg.v.mean(axis=0) 
    
    return bg

# Exercise 5.2 [5 pt]

  1. Create a large translucent sphere to represent the container [2 pt].
  2. Use a loop to create one or more small spheres to represent the atoms [3 pt].

In [7]:
def build_scene(bg):
    # --------------------------------------------------------
    # Required parameters
    # --------------------------------------------------------
    bg.rate   = 30         # Maximum frames/second
    bg.frame  = 0          # Frame counter
    bg.active = True       # Controls stopping of event loop
    bg.update = False      # Controls start/pause of update
    # --------------------------------------------------------
    bg.size = 1.05 * bg.R  # Characteristic size of scene (units of sigma) 
    bg.atom_radius = 0.1*bg.R

    # IMPORTANT: All scene widgets must go into bag.gfx
    gfx = bg.gfx
    
    # Create an empty scene
    gfx.scene = Scene('Argon Atoms\n', bg.size, height=200)

    # Create a Cartesian coordinate system with z-axis upwards
    gfx.xyz = CoordinateSystem(bg.size, up=K, draw_plane=False)

    # --------------------------------------------------------
    # Create a translucent sphere of radius bg.R,
    # centered at the origin. 
    #
    # Call the container 
    #
    #   gfx.container = ...
    # --------------------------------------------------------
    # CODE GOES HERE
    

    # --------------------------------------------------------
    # Create one or more atoms, modeled as small spheres.
    # Add each sphere to list gfx.atoms using 
    #
    #   gfx.atoms.append( ... )
    # --------------------------------------------------------
    gfx.atoms = [] # empty list
    
    for i in range(bg.N):
        pass
        # CODE GOES HERE
        
    # --------------------------------------------------------
    # Create control buttons Stop, Start/Pause and a Zoom
    # slider.
    # --------------------------------------------------------
    controls = Controls(bg)

    gfx.b_stop  = vp.button(
        text="Stop",
        background=vp.color.red,
        pos=gfx.scene.title_anchor,
        bind=controls.stop)

    gfx.b_start_pause = vp.button(
        text="Start",
        background=vp.color.green,
        pos=gfx.scene.title_anchor,
        bind=controls.start_pause)

    gfx.zoom_slider = Zoom(gfx.scene)

# Exercise 5.3 [10 pt]

Complete the function `bounce_off_wall`. 
 
But first try to understand the code snippets in the cell below. 

### README! You may find the following tips useful
  1. Distinguish between the external name of the `Bag` object, which we have called `bag`, and the name used internally, `bg`, by `build_scene`.
  2. From now on, we shall be using `numpy` for most vector operations. But `vpython` and `numpy` represent vectors differently, so be aware of the differing syntax. 
  1. `bg.r` is a numpy array of shape (N,3). It represents N position vectors, while `bg.v` represents N velocities.
  2. `bg.r[i]` accesses row $i$ of `bg.r`, that is, the position of atom $i$.
  3. The `magnitude` function computes the magnitude of one or more vectors.
  4. The `unit` function computes unit vectors from one of more vectors.

In [12]:
# 1.
a = np.array([3.0,-3.0,5.0,-2.0,7.0])
b = np.zeros_like(a) # Make an array b like a but filled with zeros

b[:] = a    # What's the difference between the operation b[:] = a and c = a?
c = a

c[3] = -42
b[3] = -99

print('1')
print('a', a)
print('b', b)
print('c', c)
print()

# 2.
print('2')
mask = a > 0
print('mask', mask)
print()

# 3.
print('3')
c = a[mask]
print('c', c)
print()

# 4.
print('4')
b[mask] = np.sqrt(c)
print('b', b)

1
a [  3.  -3.   5. -42.   7.]
b [  3.  -3.   5. -99.   7.]
c [  3.  -3.   5. -42.   7.]

2
mask [ True False  True False  True]

3
c [3. 5. 7.]

4
b [  1.73205081  -3.           2.23606798 -99.           2.64575131]


In [13]:
def bounce_off_wall(bg, eps=1.e-8):
    # ----------------------------------------------
    # 1. Determine which atoms must bounce.
    # ----------------------------------------------
    # Compute distances from the center of the
    # container.
    rmag  = magnitude(bg.r)
    # CODE HERE
    
    # ----------------------------------------------
    # 2. Determine bounce points and implement bounce.
    #    The boolean array "bounce" makes it possible
    #    to access only those elements of the arrays
    #    bg.r and bg.v that need to be changed.
    #
    # Syntax:
    #    bg.v[bounce] = ...
    #
    # Hint: See code snippets in the cell above.
    # ----------------------------------------------
    # Compute booleans "bounce"
    # CODE HERE

    if np.any(bounce):
        # Implement bounce
        # CODE GOES HERE  

        # Get current positions of atoms to bounce
        r0= bg.r0[bounce]                    # (k,3), k <= N

        # Get predicted positions and velocities
        r = bg.r[bounce]                     # (k,3)
        v = bg.v[bounce]                     # (k,3)

        # Compute unit vector from r0 to r
        # CODE HERE

        # Compute bounce points b
        # CODE HERE


        # Compute unit vector of tangent plane at bounce point.
        # CODE HERE

        # Relect velocity vector
        # CODE HERE

        # Compute distance from bounce point b to predicted point r
        # CODE HERE

        # Update r and v
        # CODE HERE

In [14]:
def propagate(bg):
    bg.r0[:] = bg.r # copy r into r0 (needed below)
    bg.r[:]  = bg.r + bg.v * bg.dt           # (N,3)

In [15]:
def update(bg):
    # ----------------------------------------------
    # 1. Compute next positions (constant
    #    velocities for now)
    # ----------------------------------------------
    propagate(bg)

    # ----------------------------------------------
    # 2. Bounce atoms off wall as needed. 
    # ----------------------------------------------
    bounce_off_wall(bg)
    
    # ----------------------------------------------
    # 3. Loop over graphical objects and update
    #    their positions. 
    # ----------------------------------------------
    for i, atom in enumerate(bg.gfx.atoms):
        x, y, z = bg.r[i]
        atom.pos.x = x
        atom.pos.y = y
        atom.pos.z = z

In [ ]:
bag = define_initial_state()

build_scene(bag)

sim = Sim(bag, update)

sim.run()

<IPython.core.display.Javascript object>

Animation ended!
